# Document Similarity — in-browser (WASM) edition

This is the **JupyterLite / Pyodide** build of the Document Similarity tool. Everything runs locally in your web browser — there is no server, no login, and your documents never leave your computer.

It uses [MinHash](https://ekzhu.com/datasketch/minhash.html) to estimate the Jaccard similarity between documents so you can find and remove duplicate or near-duplicate texts in your corpus.

> **This in-browser version is intended for small corpora and for testing.** It runs entirely in your browser tab, which is single-threaded and limited by your browser's memory, so large corpora are slow and can **crash the tab**. For larger datasets, use the Cloud (Binder) version linked in the [README](https://github.com/Australian-Text-Analytics-Platform/document-similarity#readme).

> **Your data stays private:** because everything runs locally, your documents are **never uploaded to any server** — a good fit for quick trials or sensitive data, as long as the corpus is small.

> **First run:** the cell below downloads the Python packages into your browser. This takes ~30–60 seconds the first time and is cached afterwards. Run the cells in order from top to bottom.

## 1. Setup
Run this cell first to install the required packages into the in-browser kernel.

In [ ]:
# Install packages into the Pyodide (in-browser) kernel.
# numpy / pandas / scipy / matplotlib / ipywidgets ship with Pyodide; the rest
# are fetched here. 'datasketch' is the only pure-PyPI package (MinHash/LSH).
import piplite
await piplite.install([
    'ipywidgets',
    'datasketch',
    'nltk',
    'tqdm',
    'openpyxl',
    'seaborn',
    'bokeh',
])
print('Packages installed.')

In [ ]:
# Load the Document Similarity tool and the lightweight in-browser file uploader.
print('Loading DocumentSimilarity...')
from document_similarity import DocumentSimilarity, FileUploaderWidget

ds = DocumentSimilarity()
print('Finished loading.')

## 2. Load your data

Upload one or more **.txt**, **.csv**, **.xlsx** files, or a **.zip** archive of them.

* **.txt** — each file is treated as one document.
* **.csv / .xlsx** — each row is one document. The tool uses a column named `text` (falls back to the first column) and an optional `text_name` column for the document name.

**Want to try it right away?** A few sample documents are included in the `sample_documents/` folder (two climate texts and two coffee texts are deliberate near-duplicates). Use the file picker to select them.

In [ ]:
# Display the uploader. Choose your files, then click 'Build corpus'.
loader = FileUploaderWidget()
loader

<b>Automatic deduplication of identical documents</b><br>
When you load the corpus below, the tool first finds any 100% identical documents and keeps only the first of each (alphabetically by name). The similarity analysis then runs on the remaining, non-identical texts.

In [ ]:
# Load the corpus you just built into the tool.
ds.set_text_df(loader.corpus_df)

In [ ]:
# Preview the loaded documents
ds.text_df.head()

## 3. Calculate document similarity

Set the parameters below, then run the calculation.

* **ngram_value** — number of consecutive words compared at a time (1 = word level).
* **actual_jaccard** — `False` uses fast MinHash estimation (recommended); `True` computes exact Jaccard.
* **num_perm** — number of MinHash permutations; higher = more accurate, slower.
* **similarity_cutoff** — documents at or above this score (0–1) are flagged as similar.

In [ ]:
ngram_value = 1
actual_jaccard = False  # True or False
ds.exclude_punc = True  # exclude punctuation when comparing
num_perm = 256
similarity_cutoff = 0.6  # between 0 and 1

In [ ]:
# Run the similarity calculation
ds.calculate_similarity(ngram_value, num_perm, similarity_cutoff, actual_jaccard)

## 4. Analyse similar documents

The histogram shows how many similar documents are found at each Jaccard similarity level.

In [ ]:
ds.plot_hash_similarity_by_source(ds.deduplication_df)

<b>Heatmap of similar documents</b><br>
The heatmap shows Jaccard similarity scores between pairs of similar documents (hover over a cell to see the document names). Only pairs above the cut-off are shown.

`plot_range` controls which pairs are drawn: `'y'` for all, `'n'` for none, a single number like `30`, or a range like `10-25`.

In [ ]:
plot_width = 900
plot_height = 800
font_size = '14px'
text_color = 'white'
plot_range = 'y'  # 'y' (all), 'n' (none), '30', or '10-25'

print('There are {} document pair(s) in the current process.'.format(ds.deduplication_df.shape[0]))
ds.plot_heatmap_similarity(similarity_cutoff, plot_width, plot_height, font_size, text_color, plot_range)

<b>Review and decide</b><br>
The interactive table below lists each pair of similar documents with a recommended 'keep' or 'remove' status (by default the shorter document of a pair is removed). Use the controls to view each pair side-by-side and change the decision.

In [ ]:
ds.display_deduplication_text()

## 5. Save your results

Save the documents you chose to keep (or the ones marked for removal) and download them. Click the download link that appears after you click **'Save non-duplicated texts'** or **'Save duplicated texts'**.

**This in-browser (Lite) version always saves a single `.csv`** (with `text_name` and `text` columns), regardless of how you uploaded your data. This keeps saving fast and within the browser's memory limits. For very large corpora, use the [Cloud (Binder) version](https://github.com/Australian-Text-Analytics-Platform/document-similarity#cloud-version-binder), which can also export a zip of `.txt` files to match a file-based upload and streams the download from a server.

In [ ]:
rows_to_display = 5
ds.finalise_and_save(rows_to_display)